# Ex. 22 — Google **Colab** (notícias + FAISS + RAG)

1. **Runtime → Run all** (ou célula a célula de cima para baixo).
2. **Chave Gemini:** no Colab, vá a **ícone de chaves → Secrets** e crie `GOOGLE_API_KEY` com a sua chave da [Google AI Studio](https://aistudio.google.com/apikey).  
   Se não usar Secrets, a célula seguinte pede a chave em modo oculto (`getpass`).
3. **Runtime com rede:** o host precisa de Internet (DuckDuckGo + API Gemini).
4. O índice FAISS fica em `/content/ex22_noticias_faiss/faiss_noticias_index/` durante a sessão (perde-se ao fechar o runtime, salvo que copie para o Drive).

**Nota:** este caderno é gerado por `gerar_exercicio_22_colab_ipynb.py` no repositório; após alterar os `.py`, volte a correr o script para actualizar o Colab.


In [ ]:
import subprocess, sys
pkgs = [
    "python-dotenv>=1,<2",
    "pydantic>=2.7,<3",
    "langchain-core>=0.3",
    "langchain-community>=0.3",
    "langchain-google-genai>=2",
    "langchain-text-splitters>=0.3",
    "langgraph>=0.2,<2",
    "faiss-cpu>=1.8,<2",
    "ddgs>=9,<11",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("pip ok")


In [ ]:
import base64, json, os, sys
from pathlib import Path

ROOT = Path('/content/ex22_noticias_faiss')
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

_BUNDLED_JSON = r'''
{"faiss_noticias.py": "IiIiUGVzcXVpc2EgV2ViIChEdWNrRHVja0dvKSArIGluZGV4YcOnw6NvIEZBSVNTIHBhcmEgbm90w61jaWFzIGRvIGRpYS4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBvcwppbXBvcnQgcmUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmZyb20gZG90ZW52IGltcG9ydCBsb2FkX2RvdGVudgpmcm9tIGxhbmdjaGFpbl9jb21tdW5pdHkudmVjdG9yc3RvcmVzIGltcG9ydCBGQUlTUwpmcm9tIGxhbmdjaGFpbl9jb3JlLmRvY3VtZW50cyBpbXBvcnQgRG9jdW1lbnQKZnJvbSBsYW5nY2hhaW5fZ29vZ2xlX2dlbmFpIGltcG9ydCBHb29nbGVHZW5lcmF0aXZlQUlFbWJlZGRpbmdzCmZyb20gbGFuZ2NoYWluX3RleHRfc3BsaXR0ZXJzIGltcG9ydCBSZWN1cnNpdmVDaGFyYWN0ZXJUZXh0U3BsaXR0ZXIKCgpkZWYgX2xvYWRfbG9jYWxfZW52KCkgLT4gTm9uZToKICAgIGhlcmUgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkKICAgIGZvciBkIGluIChoZXJlLnBhcmVudCwgKmhlcmUucGFyZW50cyk6CiAgICAgICAgZW52ID0gZCAvICIuZW52IgogICAgICAgIGlmIGVudi5pc19maWxlKCk6CiAgICAgICAgICAgIGxvYWRfZG90ZW52KGVudiwgb3ZlcnJpZGU9RmFsc2UpCiAgICAgICAgICAgIHJldHVybgoKCl9sb2FkX2xvY2FsX2VudigpCgojIMOabHRpbW8gdGV4dG8gYnJ1dG8gZGEgcGVzcXVpc2EgKG8gYWdlbnRlIGRlIHJlY29saGEgcHJlZW5jaGU7IG8gZGUgaW5kZXhhw6fDo28gbMOqKS4KX3VsdGltYV9wZXNxdWlzYV9icnV0YTogc3RyID0gIiIKX3VsdGltYV9jb25zdWx0YTogc3RyID0gIiIKCgpkZWYgdWx0aW1hX3Blc3F1aXNhX3NuYXBzaG90KCkgLT4gdHVwbGVbc3RyLCBzdHJdOgogICAgcmV0dXJuIF91bHRpbWFfY29uc3VsdGEsIF91bHRpbWFfcGVzcXVpc2FfYnJ1dGEKCgpkZWYgZGVmaW5pcl91bHRpbWFfcGVzcXVpc2EoY29uc3VsdGE6IHN0ciwgdGV4dG9fYnJ1dG86IHN0cikgLT4gTm9uZToKICAgIGdsb2JhbCBfdWx0aW1hX3Blc3F1aXNhX2JydXRhLCBfdWx0aW1hX2NvbnN1bHRhCiAgICBfdWx0aW1hX2NvbnN1bHRhID0gKGNvbnN1bHRhIG9yICIiKS5zdHJpcCgpCiAgICBfdWx0aW1hX3Blc3F1aXNhX2JydXRhID0gdGV4dG9fYnJ1dG8gb3IgIiIKCgpkZWYgX2dhcmFudGlyX2NoYXZlX2dvb2dsZSgpIC0+IE5vbmU6CiAgICBrID0gb3MuZW52aXJvbi5nZXQoIkdPT0dMRV9BUElfS0VZIikgb3Igb3MuZW52aXJvbi5nZXQoIkdFTUlOSV9BUElfS0VZIikKICAgIGlmIG5vdCBrOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiRGVmaW5hIEdPT0dMRV9BUElfS0VZIG91IEdFTUlOSV9BUElfS0VZIHBhcmEgZW1iZWRkaW5ncyBHZW1pbmkgKEZBSVNTKS4iKQogICAgb3MuZW52aXJvbi5zZXRkZWZhdWx0KCJHT09HTEVfQVBJX0tFWSIsIGspCgoKZGVmIF9ub3JtX2VtYmVkZGluZ19tb2RlbChub21lOiBzdHIpIC0+IHN0cjoKICAgICIiIk5vcm1hbGl6YSBvIG5vbWUgZSBtYXBlaWEgaWRlbnRpZmljYWRvcmVzIGFudGlnb3MgKDQwNCBlbSB2MWJldGEgY29tIGVtYmVkQ29udGVudCkuIiIiCiAgICBuID0gKG5vbWUgb3IgIiIpLnN0cmlwKCkKICAgIGlmIG4uc3RhcnRzd2l0aCgibW9kZWxzLyIpOgogICAgICAgIG4gPSBuLnJlbW92ZXByZWZpeCgibW9kZWxzLyIpCiAgICBsZWdhY3kgPSB7CiAgICAgICAgInRleHQtZW1iZWRkaW5nLTAwNCIsCiAgICAgICAgInRleHQtZW1iZWRkaW5nLTAwNC1lbWJlZGRpbmctcHJldmlldyIsCiAgICAgICAgImVtYmVkZGluZy0wMDEiLAogICAgfQogICAgaWYgbiBpbiBsZWdhY3k6CiAgICAgICAgcmV0dXJuICJnZW1pbmktZW1iZWRkaW5nLTAwMSIKICAgIHJldHVybiBuIG9yICJnZW1pbmktZW1iZWRkaW5nLTAwMSIKCgpkZWYgZW1iZWRkaW5nc19nZW1pbmkoKSAtPiBHb29nbGVHZW5lcmF0aXZlQUlFbWJlZGRpbmdzOgogICAgX2dhcmFudGlyX2NoYXZlX2dvb2dsZSgpCiAgICByYXcgPSAoCiAgICAgICAgb3MuZW52aXJvbi5nZXQoIkdFTUlOSV9FTUJFRERJTkdfTU9ERUwiKQogICAgICAgIG9yIG9zLmVudmlyb24uZ2V0KCJHRU1JTklfRU1CRURESU5HX01PREVMXzAwNCIpCiAgICAgICAgb3IgImdlbWluaS1lbWJlZGRpbmctMDAxIgogICAgKS5zdHJpcCgpCiAgICBtb2RlbG8gPSBfbm9ybV9lbWJlZGRpbmdfbW9kZWwocmF3KQogICAgcmV0dXJuIEdvb2dsZUdlbmVyYXRpdmVBSUVtYmVkZGluZ3MobW9kZWw9bW9kZWxvKQoKCmRlZiBwZXNxdWlzYV9kdWNrZHVja2dvX3RleHRvKGNvbnN1bHRhOiBzdHIsICosIG1heF9yZXN1bHRhZG9zOiBpbnQgPSAxMikgLT4gc3RyOgogICAgIiIiSW52b2NhIER1Y2tEdWNrR28gKGxhbmdjaGFpbl9jb21tdW5pdHkpLiBSZXF1ZXIgcmVkZS4iIiIKICAgIGZyb20gbGFuZ2NoYWluX2NvbW11bml0eS50b29scyBpbXBvcnQgRHVja0R1Y2tHb1NlYXJjaFJlc3VsdHMKCiAgICBxID0gKGNvbnN1bHRhIG9yICIiKS5zdHJpcCgpCiAgICBpZiBub3QgcToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjb25zdWx0YSB2YXppYS4iKQogICAgc2VhcmNoID0gRHVja0R1Y2tHb1NlYXJjaFJlc3VsdHMobWF4X3Jlc3VsdHM9bWF4KDEsIG1pbihtYXhfcmVzdWx0YWRvcywgMjApKSkKICAgIHRyeToKICAgICAgICByYXcgPSBzZWFyY2guaW52b2tlKHEpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHJhdyA9IHNlYXJjaC5pbnZva2UoeyJxdWVyeSI6IHF9KQogICAgdGV4dG8gPSByYXcgaWYgaXNpbnN0YW5jZShyYXcsIHN0cikgZWxzZSBzdHIocmF3KQogICAgZGVmaW5pcl91bHRpbWFfcGVzcXVpc2EocSwgdGV4dG8pCiAgICByZXR1cm4gdGV4dG8KCgpkZWYgX2ZyYWdtZW50b3NfcGFyYV9kb2N1bWVudG9zKHRleHRvOiBzdHIsIGNvbnN1bHRhOiBzdHIpIC0+IGxpc3RbRG9jdW1lbnRdOgogICAgIiIiUGFydGUgbyByZXN1bHRhZG8gRERHIGVtIGJsb2NvcyAoVVJMcy9tYW5jaGV0ZXMpIHF1YW5kbyBwb3Nzw612ZWw7IHNlbsOjbyB1bSDDum5pY28gZG9jdW1lbnRvLiIiIgogICAgdGV4dG8gPSAodGV4dG8gb3IgIiIpLnN0cmlwKCkKICAgIGlmIG5vdCB0ZXh0bzoKICAgICAgICByZXR1cm4gW10KICAgIHBhcnRlcyA9IHJlLnNwbGl0KHIiXG57Mix9IiwgdGV4dG8pCiAgICBkb2NzOiBsaXN0W0RvY3VtZW50XSA9IFtdCiAgICBmb3IgaSwgYmxvY28gaW4gZW51bWVyYXRlKHBhcnRlcyk6CiAgICAgICAgYiA9IGJsb2NvLnN0cmlwKCkKICAgICAgICBpZiBsZW4oYikgPCA0MDoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBkb2NzLmFwcGVuZCgKICAgICAgICAgICAgRG9jdW1lbnQoCiAgICAgICAgICAgICAgICBwYWdlX2NvbnRlbnQ9Yls6MTIwMDBdLAogICAgICAgICAgICAgICAgbWV0YWRhdGE9eyJjb25zdWx0YSI6IGNvbnN1bHRhLCAiYmxvY28iOiBpLCAiZm9udGUiOiAiZHVja2R1Y2tnbyJ9LAogICAgICAgICAgICApCiAgICAgICAgKQogICAgaWYgbm90IGRvY3M6CiAgICAgICAgZG9jcy5hcHBlbmQoCiAgICAgICAgICAgIERvY3VtZW50KAogICAgICAgICAgICAgICAgcGFnZV9jb250ZW50PXRleHRvWzo1MDAwMF0sCiAgICAgICAgICAgICAgICBtZXRhZGF0YT17ImNvbnN1bHRhIjogY29uc3VsdGEsICJmb250ZSI6ICJkdWNrZHVja2dvIn0sCiAgICAgICAgICAgICkKICAgICAgICApCiAgICByZXR1cm4gZG9jcwoKCmRlZiBkaXZpZGlyX2RvY3VtZW50b3MoZG9jczogbGlzdFtEb2N1bWVudF0pIC0+IGxpc3RbRG9jdW1lbnRdOgogICAgc3BsaXR0ZXIgPSBSZWN1cnNpdmVDaGFyYWN0ZXJUZXh0U3BsaXR0ZXIoY2h1bmtfc2l6ZT0xMjAwLCBjaHVua19vdmVybGFwPTE1MCkKICAgIHJldHVybiBzcGxpdHRlci5zcGxpdF9kb2N1bWVudHMoZG9jcykKCgpkZWYgY2FtaW5ob19pbmRpY2VfcGFkcmFvKCkgLT4gUGF0aDoKICAgIHJldHVybiBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50IC8gImZhaXNzX25vdGljaWFzX2luZGV4IgoKCmRlZiBpbmRleGFyX3VsdGltYV9wZXNxdWlzYV9lbV9mYWlzcygKICAgIHBlcnNpc3RfZGlyZWN0b3J5OiBQYXRoIHwgTm9uZSA9IE5vbmUsCiAgICAqLAogICAgc3Vic3RpdHVpcjogYm9vbCA9IFRydWUsCikgLT4gaW50OgogICAgIiIiSW5kZXhhIGBfdWx0aW1hX3Blc3F1aXNhX2JydXRhYCBlbSBGQUlTUy4gRGV2b2x2ZSBuw7ptZXJvIGRlIGNodW5rcy4iIiIKICAgIGNvbnN1bHRhLCBicnV0byA9IHVsdGltYV9wZXNxdWlzYV9zbmFwc2hvdCgpCiAgICBpZiBub3QgYnJ1dG8uc3RyaXAoKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoCiAgICAgICAgICAgICJOw6NvIGjDoSBwZXNxdWlzYSByZWNlbnRlIGVtIG1lbcOzcmlhLiBDaGFtZSBwcmltZWlybyBhIHRvb2wgZGUgcGVzcXVpc2EgV2ViIChvdSBgcGVzcXVpc2FfZHVja2R1Y2tnb190ZXh0b2ApLiIKICAgICAgICApCiAgICBwZXJzaXN0X2RpcmVjdG9yeSA9IHBlcnNpc3RfZGlyZWN0b3J5IG9yIGNhbWluaG9faW5kaWNlX3BhZHJhbygpCiAgICBwZXJzaXN0X2RpcmVjdG9yeSA9IFBhdGgocGVyc2lzdF9kaXJlY3RvcnkpCiAgICBwZXJzaXN0X2RpcmVjdG9yeS5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgZG9jcyA9IF9mcmFnbWVudG9zX3BhcmFfZG9jdW1lbnRvcyhicnV0bywgY29uc3VsdGEpCiAgICBjaHVua3MgPSBkaXZpZGlyX2RvY3VtZW50b3MoZG9jcykKICAgIGlmIG5vdCBjaHVua3M6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJOZW5odW0gY2h1bmsgZ2VyYWRvIGEgcGFydGlyIGRhIHBlc3F1aXNhLiIpCiAgICBlbWIgPSBlbWJlZGRpbmdzX2dlbWluaSgpCiAgICBpZiBzdWJzdGl0dWlyIGFuZCBwZXJzaXN0X2RpcmVjdG9yeS5leGlzdHMoKToKICAgICAgICBpbXBvcnQgc2h1dGlsCgogICAgICAgIGZvciBjaGlsZCBpbiBsaXN0KHBlcnNpc3RfZGlyZWN0b3J5Lml0ZXJkaXIoKSk6CiAgICAgICAgICAgIGlmIGNoaWxkLmlzX2ZpbGUoKToKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICBjaGlsZC51bmxpbmsobWlzc2luZ19vaz1UcnVlKQogICAgICAgICAgICAgICAgZXhjZXB0IE9TRXJyb3I6CiAgICAgICAgICAgICAgICAgICAgcGFzcwogICAgICAgICAgICBlbGlmIGNoaWxkLmlzX2RpcigpOgogICAgICAgICAgICAgICAgc2h1dGlsLnJtdHJlZShjaGlsZCwgaWdub3JlX2Vycm9ycz1UcnVlKQogICAgdnMgPSBGQUlTUy5mcm9tX2RvY3VtZW50cyhjaHVua3MsIGVtYikKICAgIHZzLnNhdmVfbG9jYWwoc3RyKHBlcnNpc3RfZGlyZWN0b3J5KSkKICAgIHJldHVybiBsZW4oY2h1bmtzKQoKCmRlZiBjYXJyZWdhcl9mYWlzcyhwZXJzaXN0X2RpcmVjdG9yeTogUGF0aCB8IE5vbmUgPSBOb25lKSAtPiBGQUlTUzoKICAgIHBlcnNpc3RfZGlyZWN0b3J5ID0gUGF0aChwZXJzaXN0X2RpcmVjdG9yeSBvciBjYW1pbmhvX2luZGljZV9wYWRyYW8oKSkKICAgIGlmIG5vdCBwZXJzaXN0X2RpcmVjdG9yeS5pc19kaXIoKSBvciBub3QgYW55KHBlcnNpc3RfZGlyZWN0b3J5Lml0ZXJkaXIoKSk6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKAogICAgICAgICAgICBmIsONbmRpY2UgRkFJU1MgaW5leGlzdGVudGUgZW0gYHtwZXJzaXN0X2RpcmVjdG9yeX1gLiAiCiAgICAgICAgICAgICJFeGVjdXRlIHByaW1laXJvIG8gYWdlbnRlIGRlIHJlY29saGEgKHBlc3F1aXNhICsgaW5kZXhhw6fDo28pLiIKICAgICAgICApCiAgICByZXR1cm4gRkFJU1MubG9hZF9sb2NhbCgKICAgICAgICBzdHIocGVyc2lzdF9kaXJlY3RvcnkpLAogICAgICAgIGVtYmVkZGluZ3NfZ2VtaW5pKCksCiAgICAgICAgYWxsb3dfZGFuZ2Vyb3VzX2Rlc2VyaWFsaXphdGlvbj1UcnVlLAogICAgKQoKCmRlZiByZXRyaWV2ZXJfbm90aWNpYXMoazogaW50ID0gNiwgcGVyc2lzdF9kaXJlY3Rvcnk6IFBhdGggfCBOb25lID0gTm9uZSkgLT4gQW55OgogICAgdnMgPSBjYXJyZWdhcl9mYWlzcyhwZXJzaXN0X2RpcmVjdG9yeSkKICAgIHJldHVybiB2cy5hc19yZXRyaWV2ZXIoc2VhcmNoX2t3YXJncz17ImsiOiBrfSkK", "agente_recolha_noticias.py": "IiIiQWdlbnRlIFJlQWN0OiBwZXNxdWlzYSBXZWIgKER1Y2tEdWNrR28pIGUgZ3JhdmHDp8OjbyBkbyByZXN1bHRhZG8gZW0gRkFJU1MuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmZyb20gZG90ZW52IGltcG9ydCBsb2FkX2RvdGVudgpmcm9tIGxhbmdjaGFpbl9jb3JlLm1lc3NhZ2VzIGltcG9ydCBBSU1lc3NhZ2UKZnJvbSBsYW5nY2hhaW5fY29yZS5wcm9tcHRzIGltcG9ydCBDaGF0UHJvbXB0VGVtcGxhdGUsIE1lc3NhZ2VzUGxhY2Vob2xkZXIKZnJvbSBsYW5nY2hhaW5fY29yZS50b29scyBpbXBvcnQgdG9vbApmcm9tIGxhbmdjaGFpbl9nb29nbGVfZ2VuYWkgaW1wb3J0IENoYXRHb29nbGVHZW5lcmF0aXZlQUkKZnJvbSBsYW5nZ3JhcGguY2hlY2twb2ludC5tZW1vcnkgaW1wb3J0IE1lbW9yeVNhdmVyCmZyb20gbGFuZ2dyYXBoLnByZWJ1aWx0IGltcG9ydCBjcmVhdGVfcmVhY3RfYWdlbnQKCmZyb20gZmFpc3Nfbm90aWNpYXMgaW1wb3J0IGluZGV4YXJfdWx0aW1hX3Blc3F1aXNhX2VtX2ZhaXNzLCBwZXNxdWlzYV9kdWNrZHVja2dvX3RleHRvCgoKZGVmIF9sb2FkX2xvY2FsX2VudigpIC0+IE5vbmU6CiAgICBoZXJlID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpCiAgICBmb3IgZGlyZWN0b3J5IGluIChoZXJlLnBhcmVudCwgKmhlcmUucGFyZW50cyk6CiAgICAgICAgZW52X3BhdGggPSBkaXJlY3RvcnkgLyAiLmVudiIKICAgICAgICBpZiBlbnZfcGF0aC5pc19maWxlKCk6CiAgICAgICAgICAgIGxvYWRfZG90ZW52KGVudl9wYXRoLCBvdmVycmlkZT1GYWxzZSkKICAgICAgICAgICAgcmV0dXJuCgoKX2xvYWRfbG9jYWxfZW52KCkKCgpkZWYgX21vZGVsbygpIC0+IHN0cjoKICAgIHJldHVybiAob3MuZW52aXJvbi5nZXQoIkdFTUlOSV9NT0RFTF9FWDIyIikgb3Igb3MuZW52aXJvbi5nZXQoIkdFTUlOSV9NT0RFTCIpIG9yICJnZW1pbmktMi4wLWZsYXNoIikuc3RyaXAoKQoKCmRlZiBfZ2FyYW50aXJfY2hhdmVfYXBpKCkgLT4gTm9uZToKICAgIGtleSA9IG9zLmVudmlyb24uZ2V0KCJHT09HTEVfQVBJX0tFWSIpIG9yIG9zLmVudmlyb24uZ2V0KCJHRU1JTklfQVBJX0tFWSIpCiAgICBpZiBub3Qga2V5OgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiRGVmaW5hIEdPT0dMRV9BUElfS0VZIG91IEdFTUlOSV9BUElfS0VZIG5vIGAuZW52YCBuYSByYWl6IGRvIHJlcG9zaXTDs3Jpby4iKQogICAgb3MuZW52aXJvbi5zZXRkZWZhdWx0KCJHT09HTEVfQVBJX0tFWSIsIGtleSkKCgpAdG9vbApkZWYgcGVzcXVpc2FyX25vdGljaWFzX3dlYl9wYXJhX2luZGV4YWNhbyhjb25zdWx0YTogc3RyKSAtPiBzdHI6CiAgICAiIiJQZXNxdWlzYSBuYSBXZWIgdmlhIER1Y2tEdWNrR28gKHNuaXBwZXRzIGUgbGlua3MpLiBVc2UgY29uc3VsdGFzIGN1cnRhcyBlbSBwb3J0dWd1w6pzLCBleC46IMKrbm90w61jaWFzIFBvcnR1Z2FsIGhvamXCuywgwqtlY29ub21pYSBFdXJvcGHCuy4gUmVxdWVyIHJlZGUuIiIiCiAgICByZXR1cm4gcGVzcXVpc2FfZHVja2R1Y2tnb190ZXh0byhjb25zdWx0YS5zdHJpcCgpKQoKCkB0b29sCmRlZiBncmF2YXJfdWx0aW1hX3Blc3F1aXNhX25vX2luZGljZV9mYWlzcygpIC0+IHN0cjoKICAgICIiIkluZGV4YSBlbSBGQUlTUyBvIHRleHRvIGRhICoqw7psdGltYSoqIHBlc3F1aXNhIFdlYiBmZWl0YSBuZXN0YSBzZXNzw6NvIChzdWJzdGl0dWkgbyDDrW5kaWNlIGFudGVyaW9yIG5hIHBhc3RhIGRvIGV4ZXJjw61jaW8pLiBDaGFtZSAqKmRlcG9pcyoqIGRlIGBwZXNxdWlzYXJfbm90aWNpYXNfd2ViX3BhcmFfaW5kZXhhY2FvYC4iIiIKICAgIG4gPSBpbmRleGFyX3VsdGltYV9wZXNxdWlzYV9lbV9mYWlzcyhzdWJzdGl0dWlyPVRydWUpCiAgICByZXR1cm4gZiJGQUlTUyBhY3R1YWxpemFkbzoge259IHZlY3RvcmVzIChjaHVua3MpIGdyYXZhZG9zLiIKCgpfU1lTVEVNID0gIiIiw4lzIHVtICoqZWRpdG9yIGRlIGFnw6puY2lhKiogcXVlIHByZXBhcmEgbWF0ZXJpYWwgYnJ1dG8gcGFyYSB1bSBzaXN0ZW1hIFJBRy4KCkZsdXhvOgoxLiBVc2EgYHBlc3F1aXNhcl9ub3RpY2lhc193ZWJfcGFyYV9pbmRleGFjYW9gIGNvbSAqKjEgYSA0KiogY29uc3VsdGFzIGNvbXBsZW1lbnRhcmVzIHBhcmEgY29icmlyIG8gZGlhIChwb2zDrXRpY2EsIGVjb25vbWlhLCBzb2NpZWRhZGUg4oCUIGNvbmZvcm1lIG8gcGVkaWRvIGRvIHV0aWxpemFkb3IpLgoyLiBRdWFuZG8gdGl2ZXJlcyB0cmVjaG9zIHN1ZmljaWVudGVzLCBjaGFtYSAqKnVtYSB2ZXoqKiBgZ3JhdmFyX3VsdGltYV9wZXNxdWlzYV9ub19pbmRpY2VfZmFpc3NgIHBhcmEgZ3JhdmFyICoqYXBlbmFzIGEgw7psdGltYSoqIHBlc3F1aXNhIGVtIG1lbcOzcmlhIGludGVybcOpZGlhLgoKKipBdGVuw6fDo286KiogYSB0b29sIGRlIGdyYXZhw6fDo28gaW5kZXhhIHPDsyBvIHJlc3VsdGFkbyBkYSAqKsO6bHRpbWEqKiBjaGFtYWRhIMOgIHBlc3F1aXNhLiBTZSBwcmVjaXNhcmVzIGRlIGZ1bmRpciB2w6FyaWFzIHBlc3F1aXNhcyBudW0gw7puaWNvIMOtbmRpY2UsIGZheiB1bWEgKirDumx0aW1hKiogcGVzcXVpc2EgbWFpcyBhYnJhbmdlbnRlIChleC46IMKrcmVzdW1vIG5vdMOtY2lhcyBQb3J0dWdhbCBob2plIG1hbmNoZXRlc8K7KSBlIGdyYXZhIGVtIHNlZ3VpZGEuCgpSZWdyYXM6Ci0gTsOjbyBpbnZlbnRlcyBVUkxzIG5lbSBmYWN0b3MgcXVlIG7Do28gYXBhcmXDp2FtIG5vcyBzbmlwcGV0cy4KLSBSZXNwb3N0YSBmaW5hbCBhbyB1dGlsaXphZG9yOiByZXN1bW8gY3VydG8gZG8gcXVlIGZvaSBncmF2YWRvIGUgcGFyYSBxdWUgdGVtYXMgc2VydmUgbyDDrW5kaWNlLiBQb3J0dWd1w6pzIGV1cm9wZXUuIiIiCgoKZGVmIGNyaWFyX2dyYWZvX2FnZW50ZV9yZWNvbGhhKCkgLT4gQW55OgogICAgX2dhcmFudGlyX2NoYXZlX2FwaSgpCiAgICBsbG0gPSBDaGF0R29vZ2xlR2VuZXJhdGl2ZUFJKG1vZGVsPV9tb2RlbG8oKSwgdGVtcGVyYXR1cmU9MC4yNSkKICAgIHByb21wdCA9IENoYXRQcm9tcHRUZW1wbGF0ZS5mcm9tX21lc3NhZ2VzKAogICAgICAgIFsKICAgICAgICAgICAgKCJzeXN0ZW0iLCBfU1lTVEVNKSwKICAgICAgICAgICAgTWVzc2FnZXNQbGFjZWhvbGRlcih2YXJpYWJsZV9uYW1lPSJtZXNzYWdlcyIpLAogICAgICAgIF0KICAgICkKICAgIHJldHVybiBjcmVhdGVfcmVhY3RfYWdlbnQoCiAgICAgICAgbGxtLAogICAgICAgIHRvb2xzPVtwZXNxdWlzYXJfbm90aWNpYXNfd2ViX3BhcmFfaW5kZXhhY2FvLCBncmF2YXJfdWx0aW1hX3Blc3F1aXNhX25vX2luZGljZV9mYWlzc10sCiAgICAgICAgcHJvbXB0PXByb21wdCwKICAgICAgICBjaGVja3BvaW50ZXI9TWVtb3J5U2F2ZXIoKSwKICAgICkKCgpkZWYgY29uZmlnX3JlY29saGEodGhyZWFkX2lkOiBzdHIgPSAiZXgyMi1yZWNvbGhhLW5vdGljaWFzIikgLT4gZGljdDoKICAgIHJldHVybiB7ImNvbmZpZ3VyYWJsZSI6IHsidGhyZWFkX2lkIjogdGhyZWFkX2lkfX0KCgpkZWYgdWx0aW1hX3Jlc3Bvc3RhX2Fzc2lzdGVudGUocmVzdWx0YWRvOiBkaWN0KSAtPiBzdHI6CiAgICBtc2dzID0gcmVzdWx0YWRvLmdldCgibWVzc2FnZXMiKSBvciBbXQogICAgZm9yIG0gaW4gcmV2ZXJzZWQobXNncyk6CiAgICAgICAgaWYgaXNpbnN0YW5jZShtLCBBSU1lc3NhZ2UpIGFuZCAobS5jb250ZW50IG9yICIiKS5zdHJpcCgpOgogICAgICAgICAgICBjID0gbS5jb250ZW50CiAgICAgICAgICAgIHJldHVybiBjIGlmIGlzaW5zdGFuY2UoYywgc3RyKSBlbHNlIHN0cihjKQogICAgcmV0dXJuICIiCg==", "agente_analista_noticias_rag.py": "IiIiQWdlbnRlIFJlQWN0OiBjb252ZXJzYSBzb2JyZSBub3TDrWNpYXMgaW5kZXhhZGFzIChSQUcgc29icmUgRkFJU1MpLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55Cgpmcm9tIGRvdGVudiBpbXBvcnQgbG9hZF9kb3RlbnYKZnJvbSBsYW5nY2hhaW5fY29yZS5tZXNzYWdlcyBpbXBvcnQgQUlNZXNzYWdlCmZyb20gbGFuZ2NoYWluX2NvcmUucHJvbXB0cyBpbXBvcnQgQ2hhdFByb21wdFRlbXBsYXRlLCBNZXNzYWdlc1BsYWNlaG9sZGVyCmZyb20gbGFuZ2NoYWluX2NvcmUudG9vbHMgaW1wb3J0IHRvb2wKZnJvbSBsYW5nY2hhaW5fZ29vZ2xlX2dlbmFpIGltcG9ydCBDaGF0R29vZ2xlR2VuZXJhdGl2ZUFJCmZyb20gbGFuZ2dyYXBoLmNoZWNrcG9pbnQubWVtb3J5IGltcG9ydCBNZW1vcnlTYXZlcgpmcm9tIGxhbmdncmFwaC5wcmVidWlsdCBpbXBvcnQgY3JlYXRlX3JlYWN0X2FnZW50Cgpmcm9tIGZhaXNzX25vdGljaWFzIGltcG9ydCBjYXJyZWdhcl9mYWlzcwoKCmRlZiBfbG9hZF9sb2NhbF9lbnYoKSAtPiBOb25lOgogICAgaGVyZSA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKQogICAgZm9yIGRpcmVjdG9yeSBpbiAoaGVyZS5wYXJlbnQsICpoZXJlLnBhcmVudHMpOgogICAgICAgIGVudl9wYXRoID0gZGlyZWN0b3J5IC8gIi5lbnYiCiAgICAgICAgaWYgZW52X3BhdGguaXNfZmlsZSgpOgogICAgICAgICAgICBsb2FkX2RvdGVudihlbnZfcGF0aCwgb3ZlcnJpZGU9RmFsc2UpCiAgICAgICAgICAgIHJldHVybgoKCl9sb2FkX2xvY2FsX2VudigpCgoKZGVmIF9tb2RlbG8oKSAtPiBzdHI6CiAgICByZXR1cm4gKG9zLmVudmlyb24uZ2V0KCJHRU1JTklfTU9ERUxfRVgyMiIpIG9yIG9zLmVudmlyb24uZ2V0KCJHRU1JTklfTU9ERUwiKSBvciAiZ2VtaW5pLTIuMC1mbGFzaCIpLnN0cmlwKCkKCgpkZWYgX2dhcmFudGlyX2NoYXZlX2FwaSgpIC0+IE5vbmU6CiAgICBrZXkgPSBvcy5lbnZpcm9uLmdldCgiR09PR0xFX0FQSV9LRVkiKSBvciBvcy5lbnZpcm9uLmdldCgiR0VNSU5JX0FQSV9LRVkiKQogICAgaWYgbm90IGtleToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkRlZmluYSBHT09HTEVfQVBJX0tFWSBvdSBHRU1JTklfQVBJX0tFWSBubyBgLmVudmAgbmEgcmFpeiBkbyByZXBvc2l0w7NyaW8uIikKICAgIG9zLmVudmlyb24uc2V0ZGVmYXVsdCgiR09PR0xFX0FQSV9LRVkiLCBrZXkpCgoKQHRvb2wKZGVmIGNvbnN1bHRhcl9pbmRpY2Vfbm90aWNpYXNfZmFpc3MocGVyZ3VudGFfcmVjdXBlcmFjYW86IHN0cikgLT4gc3RyOgogICAgIiIiUmVjdXBlcmEgdHJlY2hvcyBkbyDDrW5kaWNlIEZBSVNTIChzZW3Dom50aWNhKS4gRm9ybXVsZSB1bWEgcGVyZ3VudGEgY3VydGEgb3UgcGFsYXZyYXMtY2hhdmUgYWxpbmhhZGFzIGNvbSBvIHF1ZSBwcm9jdXJhIChleC46IMKraW1wYWN0byB0YXhhcyBqdXJvc8K7LCDCq2VsZWnDp8O1ZXMgYXV0w6FycXVpY2FzwrspLiIiIgogICAgcSA9IChwZXJndW50YV9yZWN1cGVyYWNhbyBvciAiIikuc3RyaXAoKQogICAgaWYgbm90IHE6CiAgICAgICAgcmV0dXJuICIocGVyZ3VudGEgdmF6aWEpIgogICAgdnMgPSBjYXJyZWdhcl9mYWlzcygpCiAgICBkb2NzID0gdnMuc2ltaWxhcml0eV9zZWFyY2gocSwgaz04KQogICAgYmxvY29zOiBsaXN0W3N0cl0gPSBbXQogICAgZm9yIGksIGQgaW4gZW51bWVyYXRlKGRvY3MsIDEpOgogICAgICAgIG1ldGEgPSBkLm1ldGFkYXRhIG9yIHt9CiAgICAgICAgY2FiID0gZiItLS0gVHJlY2hvIHtpfSAoY29uc3VsdGEgb3JpZ2luYWwgZGEgcmVjb2xoYToge21ldGEuZ2V0KCdjb25zdWx0YScsICduL2QnKX0pXG4iCiAgICAgICAgYmxvY29zLmFwcGVuZChjYWIgKyAoZC5wYWdlX2NvbnRlbnQgb3IgIiIpLnN0cmlwKCkpCiAgICB0ZXh0byA9ICJcblxuIi5qb2luKGJsb2NvcykKICAgIHJldHVybiB0ZXh0b1s6MjgwMDBdCgoKX1NZU1RFTSA9ICIiIsOJcyB1bSAqKmFuYWxpc3RhIGVkaXRvcmlhbCoqIHF1ZSBjb21lbnRhICoqYXBlbmFzKiogbyBxdWUgY29uc3RhciBkb3MgdHJlY2hvcyBkZXZvbHZpZG9zIHBlbGEgZmVycmFtZW50YSBgY29uc3VsdGFyX2luZGljZV9ub3RpY2lhc19mYWlzc2AuCgpBbnRlcyBkZSBjb25jbHVzw7VlcyBvdSDCq2xlaXR1cmFzIGRvIGRpYcK7LCAqKmludm9jYSBhIGZlcnJhbWVudGEqKiBjb20gc3ViLXBlcmd1bnRhcyBkaWZlcmVudGVzIHNlIHByZWNpc2FyZXMgZGUgbWFpcyBjb250ZXh0byAoZWNvbm9taWEsIHBvbMOtdGljYSBleHRlcm5hLCBzb2NpZWRhZGUsIGV0Yy4pLgoKUmVncmFzOgotIENpdGEgbGFjdW5hczogbyDDrW5kaWNlIHBvZGUgc2VyIGluY29tcGxldG8gb3UgZGVzYWN0dWFsaXphZG8gZW0gcmVsYcOnw6NvIMOgIGhvcmEgYWN0dWFsLgotIERlc3RhY2EgKippbXBsaWNhw6fDtWVzKiogZSAqKnJpc2NvcyoqIGVtIGxpbmd1YWdlbSBjbGFyYSAocG9ydHVndcOqcyBldXJvcGV1KSwgc2VtIHNlbnNhY2lvbmFsaXNtby4KLSBTZSBvIHV0aWxpemFkb3IgcGVkaXIgZmFjdG9zIHF1ZSBuw6NvIGFwYXJlw6dhbSBub3MgdHJlY2hvcywgZGl6IGV4cGxpY2l0YW1lbnRlIHF1ZSBvIMOtbmRpY2UgbsOjbyBjb250w6ltIGVzc2EgaW5mb3JtYcOnw6NvLgotIE9wY2lvbmFsbWVudGUgdGVybWluYSBjb20gKiozIHBlcmd1bnRhcyBkZSBzZWd1aW1lbnRvKiogw7p0ZWlzIHBhcmEgYXByb2Z1bmRhciAoYWluZGEgZGVudHJvIGRvIFJBRykuIiIiCgoKZGVmIGNyaWFyX2dyYWZvX2FnZW50ZV9hbmFsaXN0YSgpIC0+IEFueToKICAgIF9nYXJhbnRpcl9jaGF2ZV9hcGkoKQogICAgbGxtID0gQ2hhdEdvb2dsZUdlbmVyYXRpdmVBSShtb2RlbD1fbW9kZWxvKCksIHRlbXBlcmF0dXJlPTAuMzUpCiAgICBwcm9tcHQgPSBDaGF0UHJvbXB0VGVtcGxhdGUuZnJvbV9tZXNzYWdlcygKICAgICAgICBbCiAgICAgICAgICAgICgic3lzdGVtIiwgX1NZU1RFTSksCiAgICAgICAgICAgIE1lc3NhZ2VzUGxhY2Vob2xkZXIodmFyaWFibGVfbmFtZT0ibWVzc2FnZXMiKSwKICAgICAgICBdCiAgICApCiAgICByZXR1cm4gY3JlYXRlX3JlYWN0X2FnZW50KAogICAgICAgIGxsbSwKICAgICAgICB0b29scz1bY29uc3VsdGFyX2luZGljZV9ub3RpY2lhc19mYWlzc10sCiAgICAgICAgcHJvbXB0PXByb21wdCwKICAgICAgICBjaGVja3BvaW50ZXI9TWVtb3J5U2F2ZXIoKSwKICAgICkKCgpkZWYgY29uZmlnX2FuYWxpc3RhKHRocmVhZF9pZDogc3RyID0gImV4MjItYW5hbGlzdGEtcmFnIikgLT4gZGljdDoKICAgIHJldHVybiB7ImNvbmZpZ3VyYWJsZSI6IHsidGhyZWFkX2lkIjogdGhyZWFkX2lkfX0KCgpkZWYgdWx0aW1hX3Jlc3Bvc3RhX2Fzc2lzdGVudGUocmVzdWx0YWRvOiBkaWN0KSAtPiBzdHI6CiAgICBtc2dzID0gcmVzdWx0YWRvLmdldCgibWVzc2FnZXMiKSBvciBbXQogICAgZm9yIG0gaW4gcmV2ZXJzZWQobXNncyk6CiAgICAgICAgaWYgaXNpbnN0YW5jZShtLCBBSU1lc3NhZ2UpIGFuZCAobS5jb250ZW50IG9yICIiKS5zdHJpcCgpOgogICAgICAgICAgICBjID0gbS5jb250ZW50CiAgICAgICAgICAgIHJldHVybiBjIGlmIGlzaW5zdGFuY2UoYywgc3RyKSBlbHNlIHN0cihjKQogICAgcmV0dXJuICIiCg=="}
'''
_BUNDLED = json.loads(_BUNDLED_JSON)
for fn, b64 in _BUNDLED.items():
    (ROOT / fn).write_bytes(base64.b64decode(b64.encode('ascii')))

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Módulos em', ROOT, ':', sorted(p.name for p in ROOT.glob('*.py')))


In [ ]:
import os
try:
    from google.colab import userdata
    _k = userdata.get("GOOGLE_API_KEY")
except Exception:
    _k = None
if not (_k or "").strip():
    from getpass import getpass
    _k = getpass("Cole a GOOGLE_API_KEY (Gemini): ")
os.environ["GOOGLE_API_KEY"] = (_k or "").strip()
os.environ.setdefault("GEMINI_MODEL", "gemini-2.0-flash")
print("Chave definida:", "sim" if os.environ["GOOGLE_API_KEY"] else "não")


In [ ]:
from langchain_core.messages import HumanMessage

from agente_recolha_noticias import (
    config_recolha,
    criar_grafo_agente_recolha,
    ultima_resposta_assistente as ultima_recolha,
)

g_recolha = criar_grafo_agente_recolha()
cfg = config_recolha("colab-recolha")
msg = (
    "Pesquisa notícias de Portugal para hoje (manchetes e economia) "
    "e grava o índice FAISS. Se precisares, faz uma última consulta abrangente antes de gravar."
)
out = g_recolha.invoke({"messages": [HumanMessage(content=msg)]}, cfg)
print(ultima_recolha(out))


In [ ]:
from langchain_core.messages import HumanMessage

from agente_analista_noticias_rag import (
    config_analista,
    criar_grafo_agente_analista,
    ultima_resposta_assistente as ultima_analista,
)

g_rag = criar_grafo_agente_analista()
cfg2 = config_analista("colab-rag")
pergunta = (
    "Com base nas notícias indexadas, resume os temas principais e indica "
    "dois riscos ou incertezas para um decisor. Usa a ferramenta de recuperação."
)
out2 = g_rag.invoke({"messages": [HumanMessage(content=pergunta)]}, cfg2)
print(ultima_analista(out2))
